# 01 — Stock Market Data Collection
**Project:** Reinforcement Learning with RNN-Based Market State Representation for Algorithmic Trading  
**Step:** Download 5 years of daily OHLCV data for 5 NSE equities via Yahoo Finance.

> **Note:** This notebook requires network access to download data. The raw CSVs already exist in `data/raw/`.
> Re-running this notebook will overwrite them with freshly downloaded data. Skip execution if the raw data is already present.

## 1. Imports & Configuration

In [ ]:
import os
import sys
import logging

import pandas as pd
import yfinance as yf

# Set working directory to project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

## 2. Stock Universe & Paths

In [ ]:
# NSE Ticker -> Output filename base
TARGET_STOCKS = {
    "RELIANCE.NS": "RELIANCE",
    "TCS.NS":      "TCS",
    "INFY.NS":     "INFY",
    "HDFCBANK.NS": "HDFCBANK",
    "ITC.NS":      "ITC",
}

RAW_DATA_DIR = os.path.join("data", "raw")
REQUIRED_COLUMNS = ["Date", "Open", "High", "Low", "Close", "Volume"]

print(f"Raw data directory : {RAW_DATA_DIR}")
print(f"Target stocks      : {list(TARGET_STOCKS.keys())}")

## 3. Download Function

In [ ]:
def fetch_stock_raw_data(ticker: str, period: str = "5y") -> pd.DataFrame:
    """
    Downloads historical market data from Yahoo Finance for a given ticker.

    Args:
        ticker: Yahoo Finance ticker symbol (e.g. 'RELIANCE.NS').
        period: Data period to collect (default '5y').

    Returns:
        pd.DataFrame: Cleaned raw OHLCV data with Date column.

    Raises:
        RuntimeError: If download fails or returns an empty DataFrame.
    """
    logger.info(f"Downloading {ticker} (period={period}) ...")
    try:
        df = yf.download(
            tickers=ticker,
            period=period,
            interval="1d",
            auto_adjust=False,
            progress=False,
            multi_level_index=False,   # yfinance >= 0.2.x compatibility
        )
    except Exception as e:
        raise RuntimeError(f"Network/API exception downloading {ticker}: {e}") from e

    if df is None or df.empty:
        raise RuntimeError(f"No data returned for '{ticker}' (period={period}).")

    df = df.reset_index()

    if "Date" not in df.columns:
        raise RuntimeError(f"Missing 'Date' in download for {ticker}. Found: {df.columns.tolist()}")

    # Normalize Date to timezone-naive YYYY-MM-DD string
    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None).dt.strftime("%Y-%m-%d")

    # Verify OHLCV columns exist
    missing = [c for c in ["Open", "High", "Low", "Close", "Volume"] if c not in df.columns]
    if missing:
        raise RuntimeError(f"'{ticker}' missing columns: {missing}")

    # Retain only required columns in canonical order
    df = df[REQUIRED_COLUMNS].copy()

    # Ensure chronological ordering
    df["_dt"] = pd.to_datetime(df["Date"])
    df = df.sort_values("_dt").drop(columns=["_dt"]).reset_index(drop=True)

    logger.info(
        f"Fetched {ticker}: {len(df)} rows  "
        f"[{df['Date'].iloc[0]} → {df['Date'].iloc[-1]}]"
    )
    return df

## 4. Download All Stocks & Save Raw CSVs

In [ ]:
os.makedirs(RAW_DATA_DIR, exist_ok=True)
saved = {}
errors = []

print("=" * 60)
print("STEP 1: Stock Market Data Collection (Yahoo Finance – NSE)")
print("=" * 60)

for ticker, name in TARGET_STOCKS.items():
    out_path = os.path.join(RAW_DATA_DIR, f"{name}.csv")
    try:
        raw_df = fetch_stock_raw_data(ticker=ticker, period="5y")
        raw_df.to_csv(out_path, index=False)
        saved[ticker] = out_path
        logger.info(f"Saved {ticker} → {out_path}  ({os.path.getsize(out_path):,} bytes)")
    except Exception as e:
        logger.error(f"Failed {ticker}: {e}")
        errors.append(f"{ticker}: {e}")

if errors:
    print("\nERRORS:")
    for err in errors:
        print(f"  {err}")
else:
    print(f"\nData collection complete. {len(saved)}/{len(TARGET_STOCKS)} stocks saved.")

## 5. Verification – Raw File Shapes

In [ ]:
print("\nRaw Data Verification:")
print(f"{'Stock':<12} {'Rows':>6}  {'Cols':>5}  {'Start Date':>12}  {'End Date':>12}  {'Size (bytes)':>13}")
print("-" * 70)

for ticker, name in TARGET_STOCKS.items():
    path = os.path.join(RAW_DATA_DIR, f"{name}.csv")
    if os.path.exists(path):
        df = pd.read_csv(path)
        size = os.path.getsize(path)
        print(
            f"{name:<12} {len(df):>6}  {len(df.columns):>5}  "
            f"{df['Date'].min():>12}  {df['Date'].max():>12}  {size:>13,}"
        )
    else:
        print(f"{name:<12} *** MISSING ***")